# Arquitetura Cliente-Servidor — Exemplo Executável

**Guia Aberto de Arquitetura de Software** · [Página do tópico](https://ifpe-belojardim-es.github.io/guia-arquitetura-de-software/content/04-cliente-servidor/)

Autores: Edson Cavalcante Valença e Matheus Emanuel · Licença: CC BY-NC 4.0

---

Este notebook demonstra na prática o funcionamento da **Arquitetura Cliente-Servidor**:

1. **Rodar de ponta a ponta em ambiente limpo** — utiliza bibliotecas nativas da linguagem Python (sem necessidade de instalação externa).
2. **Demonstrar o conceito** — servidor centralizando regras de negócio e acesso ao banco de dados com clientes realizando requisições via HTTP/API.
3. **Mostrar a violação** — um cliente ignorando o servidor e tentando acessar/modificar o banco de dados diretamente.
4. **Medir a consequência** — demonstrando como o acesso direto causa corrupção de regras de negócio e falhas por bloqueio de concorrência (`database locked`).

## 0. Dependências

Utilizamos apenas módulos da biblioteca padrão do Python para garantir que o notebook seja 100% executável em qualquer ambiente (como Google Colab ou Jupyter local) sem necessidade de `pip install`.

In [ ]:
import sys
import json
import sqlite3
import threading
import time
from http.server import HTTPServer, BaseHTTPRequestHandler
from urllib.request import urlopen, Request
from urllib.error import HTTPError

print(f"Python {sys.version.split()[0]}")

## 1. O problema

Em um sistema de e-commerce e controle de estoque, múltiplos clientes (aplicativos mobile, navegadores web) precisam consultar e realizar pedidos. 

Se cada cliente acessasse o banco de dados diretamente, as regras de negócio (como verificação de saldo de estoque e cálculo de total) teriam que ser duplicadas em cada cliente, e as credenciais do banco ficariam expostas no dispositivo do usuário.

A solução **Cliente-Servidor** estabelece uma camada centralizada (o Servidor) que responde a requisições via rede, valida permissões, executa regras de negócio e gerencia o acesso ao banco de dados.

## 2. Implementação seguindo o estilo

Abaixo criamos o banco de dados do servidor, um servidor HTTP centralizado para orquestrar as compras e uma função cliente que interage estritamente via requisições HTTP (`POST /pedidos`).

In [ ]:
# --- 1. PREPARAÇÃO DO BANCO DE DADOS CENTRAL (PRIVADO DO SERVIDOR) ---
def inicializar_banco_servidor():
    conn = sqlite3.connect("servidor_loja.db")
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS estoque (
            produto_id INTEGER PRIMARY KEY,
            nome TEXT,
            quantidade INTEGER,
            preco REAL
        )
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS pedidos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            produto_id INTEGER,
            quantidade INTEGER,
            total REAL
        )
    """)
    cursor.execute("DELETE FROM estoque")
    cursor.execute("DELETE FROM pedidos")
    cursor.execute("INSERT INTO estoque VALUES (1, 'Teclado Mecânico', 10, 250.00)")
    conn.commit()
    conn.close()

inicializar_banco_servidor()

# --- 2. SERVIDOR CENTRALIZADO (HTTP REST API) ---
class ServidorPedidoHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        if self.path == "/pedidos":
            content_length = int(self.headers['Content-Length'])
            post_data = json.loads(self.rfile.read(content_length).decode('utf-8'))
            
            produto_id = post_data.get("produto_id")
            qtd_desejada = post_data.get("quantidade", 1)

            # Processamento de regra de negócio centralizado no SERVIDOR
            conn = sqlite3.connect("servidor_loja.db")
            cursor = conn.cursor()
            cursor.execute("SELECT nome, quantidade, preco FROM estoque WHERE produto_id = ?", (produto_id,))
            item = cursor.fetchone()

            if item and item[1] >= qtd_desejada:
                novo_estoque = item[1] - qtd_desejada
                cursor.execute("UPDATE estoque SET quantidade = ? WHERE produto_id = ?", (novo_estoque, produto_id))
                total = item[2] * qtd_desejada
                cursor.execute("INSERT INTO pedidos (produto_id, quantidade, total) VALUES (?, ?, ?)", 
                               (produto_id, qtd_desejada, total))
                conn.commit()
                conn.close()

                resposta = {"status": "sucesso", "mensagem": "Pedido realizado!", "total": total}
                self.send_response(201)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(json.dumps(resposta).encode('utf-8'))
            else:
                conn.close()
                resposta = {"status": "erro", "mensagem": "Estoque insuficiente ou produto inexistente"}
                self.send_response(400)
                self.send_header("Content-Type", "application/json")
                self.end_headers()
                self.wfile.write(json.dumps(resposta).encode('utf-8'))
    
    def log_message(self, format, *args):
        return  # Silencia logs padrao para manter o output limpo

# Inicializa o servidor HTTP em background (Daemon Thread)
httpd = HTTPServer(('127.0.0.1', 8085), ServidorPedidoHandler)
thread_servidor = threading.Thread(target=httpd.serve_forever, daemon=True)
thread_servidor.start()
print("Servidor central rodando na porta 8085...")

### Testando a comunicação Cliente-Servidor

O cliente faz solicitações via HTTP. O servidor valida a regra de negócio e retorna o resultado estruturado.

In [ ]:
def cliente_fazer_pedido(produto_id: int, quantidade: int):
    url = "http://127.0.0.1:8085/pedidos"
    payload = json.dumps({"produto_id": produto_id, "quantidade": quantidade}).encode('utf-8')
    req = Request(url, data=payload, headers={'Content-Type': 'application/json'})
    
    try:
        with urlopen(req) as response:
            res_data = json.loads(response.read().decode('utf-8'))
            print(f"[Cliente HTTP] Sucesso (Status 201): {res_data}")
    except HTTPError as e:
        res_data = json.loads(e.read().decode('utf-8'))
        print(f"[Cliente HTTP] Erro (Status {e.code}): {res_data}")

# Compra válida
cliente_fazer_pedido(1, 2)

# Compra excedendo o estoque disponível
cliente_fazer_pedido(1, 50)

## 3. A violação

A violação ocorre quando um aplicativo cliente ignora a API do servidor e estabelece uma conexão direta com o banco de dados central.

In [ ]:
# ANTIPADRÃO: O cliente insere diretamente no banco de dados sem passar pelo servidor
def cliente_violador_inserir_direto(produto_id: int, quantidade: int, valor_inventado: float):
    print("\n[Cliente Violador] Bypassing servidor e gravando direto na base...")
    conn = sqlite3.connect("servidor_loja.db")
    cursor = conn.cursor()
    # Insere pedido sem checar estoque nem calcular preço correto!
    cursor.execute("INSERT INTO pedidos (produto_id, quantidade, total) VALUES (?, ?, ?)", 
                   (produto_id, quantidade, valor_inventado))
    conn.commit()
    conn.close()
    print("[Cliente Violador] Dados inseridos com sucesso sem passar por nenhuma regra de negócio!")

cliente_violador_inserir_direto(1, 999, -500.00)

## 4. Consequência mensurável

Abaixo medimos dois impactos claros da violação:
1. **Inconsistência/Corrupção de Dados:** Registros inválidos (valores negativos ou estoques estourados) gravados no banco.
2. **Concorrência e Bloqueios:** Múltiplas conexões de clientes disputando acesso direto ao arquivo do banco causam falhas de contenção (`database locked`).

In [ ]:
print("--- VERIFICAÇÃO DO ESTADO DO BANCO DE DADOS ---")
conn = sqlite3.connect("servidor_loja.db")
cursor = conn.cursor()
cursor.execute("SELECT * FROM pedidos")
pedidos = cursor.fetchall()

print(f"Total de pedidos registrados: {len(pedidos)}")
for p in pedidos:
    print(f"  -> Pedido ID {p[0]}: Produto {p[1]}, Qtd {p[2]}, Total: R$ {p[3]:.2f}")

# Simulação de múltiplas conexões diretas concorrentes
print("\n--- TESTE DE CONCORRÊNCIA COM CONEXÕES DIRETAS --- ")
falhas_bloqueio = 0

def operacao_direta_concorrente():
    global falhas_bloqueio
    try:
        c = sqlite3.connect("servidor_loja.db", timeout=0.001)
        cur = c.cursor()
        cur.execute("BEGIN EXCLUSIVE")
        time.sleep(0.01)
        c.commit()
        c.close()
    except sqlite3.OperationalError:
        falhas_bloqueio += 1

threads = [threading.Thread(target=operacao_direta_concorrente) for _ in range(25)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"Conexões diretas rejeitadas por bloqueio (Database Locked): {falhas_bloqueio} de 25")
conn.close()

## 5. Conclusão

A mediação realizada pelo servidor garante a integridade dos dados, a aplicação consistente das regras de negócio e o gerenciamento centralizado do banco de dados, protegendo o sistema contra inconsistências e concorrência desordenada.

---

### Referências

[1] Tanenbaum, A. S., Van Steen, M. *Sistemas Distribuídos: Princípios e Paradigmas*, 2ª ed. Pearson, 2007.

[2] Bass, L., Clements, P., Kazman, R. *Software Architecture in Practice*, 4ª ed. Addison-Wesley, 2021.

[3] Sommerville, I. *Engenharia de Software*, 10ª ed. Pearson, 2019.

---

Conteúdo sob CC BY-NC 4.0 — uso livre com crédito. Código sob MIT.